In [1]:
import pandas as pd

# Assuming train_df and test_df are the DataFrames from the previous step
train_df = pd.read_csv('D:\\LLM-Driven_AI-Studio\\MLAgent\\data\\benchmark/DSEval/datasets/06_employee/train.csv')
test_df = pd.read_csv('D:\\LLM-Driven_AI-Studio\\MLAgent\\data\\benchmark/DSEval/datasets/06_employee/test.csv')

# Copy the DataFrames to avoid modifying the original data
train_df_copy = train_df.copy()
test_df_copy = test_df.copy()

# Feature construction: Calculate 'YearsInCompany'
train_df_copy['YearsInCompany'] = 2023 - train_df_copy['JoiningYear']
test_df_copy['YearsInCompany'] = 2023 - test_df_copy['JoiningYear']

# Display the first few rows of the modified DataFrames
train_df_copy.head(), test_df_copy.head()


(   Education  JoiningYear  ... LeaveOrNot  YearsInCompany
 0    Masters         2013  ...          1              10
 1  Bachelors         2012  ...          0              11
 2    Masters         2017  ...          0               6
 3    Masters         2012  ...          0              11
 4    Masters         2017  ...          0               6
 
 [5 rows x 10 columns],
    Education  JoiningYear  ... LeaveOrNot  YearsInCompany
 0  Bachelors         2016  ...          1               7
 1  Bachelors         2013  ...          0              10
 2  Bachelors         2017  ...          1               6
 3    Masters         2015  ...          0               8
 4  Bachelors         2012  ...          0              11
 
 [5 rows x 10 columns])

In [2]:
from metagpt.tools.libs.data_preprocess import get_column_info

# Get column info for the modified training and test datasets
train_column_info = get_column_info(train_df_copy)
test_column_info = get_column_info(test_df_copy)

print("Train dataset column_info:")
print(train_column_info)
print("\nTest dataset column_info:")
print(test_column_info)


2025-08-30 18:52:09.238 | INFO     | metagpt.const:get_metagpt_package_root:29 - Package root set to D:\LLM-Driven_AI-Studio\MLAgent\experiments\DataInterpreter


Train dataset column_info:
{'Category': ['Education', 'City', 'Gender', 'EverBenched'], 'Numeric': ['JoiningYear', 'PaymentTier', 'Age', 'ExperienceInCurrentDomain', 'LeaveOrNot', 'YearsInCompany'], 'Datetime': [], 'Others': []}

Test dataset column_info:
{'Category': ['Education', 'City', 'Gender', 'EverBenched'], 'Numeric': ['JoiningYear', 'PaymentTier', 'Age', 'ExperienceInCurrentDomain', 'LeaveOrNot', 'YearsInCompany'], 'Datetime': [], 'Others': []}


In [3]:
from metagpt.tools.libs.data_preprocess import OneHotEncode

# Initialize the OneHotEncode tool with the categorical columns
one_hot_encoder = OneHotEncode(features=['Education', 'City', 'Gender', 'EverBenched'])

# Fit the encoder on the training data and transform both the training and test data
train_df_encoded = one_hot_encoder.fit_transform(train_df_copy)
test_df_encoded = one_hot_encoder.transform(test_df_copy)

# Display the first few rows of the encoded datasets
train_df_encoded.head(), test_df_encoded.head()


D:\LLM-Driven_AI-Studio\MLAgent\experiments\DataInterpreter\venv\lib\site-packages\sklearn\preprocessing\_encoders.py:975: FutureWarning: `sparse` was renamed to `sparse_output` in version 1.2 and will be removed in 1.4. `sparse_output` is ignored unless you leave `sparse` to its default value.
  warnings.warn(


(   JoiningYear  PaymentTier  Age  ...  Gender_Male  EverBenched_No  EverBenched_Yes
 0         2013            3   30  ...          1.0             1.0              0.0
 1         2012            3   25  ...          1.0             1.0              0.0
 2         2017            2   29  ...          1.0             1.0              0.0
 3         2012            3   24  ...          1.0             1.0              0.0
 4         2017            2   35  ...          1.0             1.0              0.0
 
 [5 rows x 16 columns],
    JoiningYear  PaymentTier  Age  ...  Gender_Male  EverBenched_No  EverBenched_Yes
 0         2016            3   24  ...          0.0             1.0              0.0
 1         2013            3   26  ...          0.0             1.0              0.0
 2         2017            2   25  ...          0.0             1.0              0.0
 3         2015            2   28  ...          0.0             1.0              0.0
 4         2012            3   33  ... 

In [4]:
from metagpt.tools.libs.data_preprocess import get_column_info

# Using the latest DataFrame from the finished tasks
column_info_train = get_column_info(train_df_encoded)
column_info_test = get_column_info(test_df_encoded)

print("Column Info for Train DataFrame")
print(column_info_train)
print("\nColumn Info for Test DataFrame")
print(column_info_test)


Column Info for Train DataFrame
{'Category': [], 'Numeric': ['JoiningYear', 'PaymentTier', 'Age', 'ExperienceInCurrentDomain', 'LeaveOrNot', 'YearsInCompany', 'Education_Bachelors', 'Education_Masters', 'Education_PHD', 'City_Bangalore', 'City_New Delhi', 'City_Pune', 'Gender_Female', 'Gender_Male', 'EverBenched_No', 'EverBenched_Yes'], 'Datetime': [], 'Others': []}

Column Info for Test DataFrame
{'Category': [], 'Numeric': ['JoiningYear', 'PaymentTier', 'Age', 'ExperienceInCurrentDomain', 'LeaveOrNot', 'YearsInCompany', 'Education_Bachelors', 'Education_Masters', 'Education_PHD', 'City_Bangalore', 'City_New Delhi', 'City_Pune', 'Gender_Female', 'Gender_Male', 'EverBenched_No', 'EverBenched_Yes'], 'Datetime': [], 'Others': []}


In [5]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, confusion_matrix

# Define the feature columns and target column
feature_columns = ['JoiningYear', 'PaymentTier', 'Age', 'ExperienceInCurrentDomain', 'YearsInCompany', 
                   'Education_Bachelors', 'Education_Masters', 'Education_PHD', 
                   'City_Bangalore', 'City_New Delhi', 'City_Pune', 
                   'Gender_Female', 'Gender_Male', 'EverBenched_No', 'EverBenched_Yes']
target_column = 'LeaveOrNot'

# Initialize the Random Forest Classifier
rf_classifier = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)

# Train the model
rf_classifier.fit(train_df_encoded[feature_columns], train_df_encoded[target_column])

# Make predictions on the test set
test_predictions = rf_classifier.predict(test_df_encoded[feature_columns])
test_probabilities = rf_classifier.predict_proba(test_df_encoded[feature_columns])[:, 1]

# Compute the area under the ROC curve
roc_auc = roc_auc_score(test_df_encoded[target_column], test_probabilities)

# Compute the confusion matrix
conf_matrix = confusion_matrix(test_df_encoded[target_column], test_predictions)

roc_auc, conf_matrix


(0.9039425974158624,
 array([[592,  18],
        [ 98, 223]], dtype=int64))